In [4]:
# Stage-2: Apply authoritative college mapping (correct paths for Stage1EDA.ipynb)
import json
from pathlib import Path
import pandas as pd

# Notebook is in Version2/notebooks/
NOTEBOOK_DIR = Path(".").resolve()           # Version2/notebooks
BASE = NOTEBOOK_DIR.parent                   # Version2/
DATA_DIR = BASE / "data" / "processed_data"  # Version2/data/processed_data/

INPUT_CSV = DATA_DIR / "KCET_split_cleaned.csv"
MAPPING_JSON = DATA_DIR / "college_mapping_2024.json"

OUTPUT_CSV = DATA_DIR / "KCET_split_mapped.csv"
UNMAPPED_FILE = DATA_DIR / "kcet_unmapped_college_codes.csv"
REPORT_FILE = DATA_DIR / "kcet_college_mapping_report.csv"

print("Notebook Directory:", NOTEBOOK_DIR)
print("Base Directory:", BASE)
print("Data Directory:", DATA_DIR)

# 1) Load the cleaned CSV
print("\nLoading CSV:", INPUT_CSV)
df = pd.read_csv(INPUT_CSV, dtype=str, low_memory=False)
print("Rows, columns:", df.shape)

# 2) Load official mapping JSON
print("Loading JSON mapping:", MAPPING_JSON)
with open(MAPPING_JSON, "r", encoding="utf-8") as f:
    college_map = json.load(f)

# Normalize JSON mappings
college_map = {str(k).strip(): str(v).strip() for k, v in college_map.items()}

# 3) Check required field
if "College_Code" not in df.columns:
    raise KeyError("ERROR: 'College_Code' column missing in dataset.")

# 4) Apply mapping
df["College_Code"] = df["College_Code"].astype(str).str.strip()
df["College_Name_mapped"] = df["College_Code"].map(college_map)

total_rows = len(df)
n_mapped = df["College_Name_mapped"].notna().sum()
n_unmapped = df["College_Name_mapped"].isna().sum()

# Backup original name if present
if "College_Name" in df.columns:
    df["College_Name_original"] = df["College_Name"]

df["College_Name"] = df["College_Name_mapped"]

# 5) Save mapped CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nMapped CSV saved → {OUTPUT_CSV}")

# 6) Save unmapped codes
if n_unmapped > 0:
    unmapped_df = (
        df[df["College_Name"].isna()]
        .groupby("College_Code")
        .size()
        .reset_index(name="rows_count")
        .sort_values("rows_count", ascending=False)
    )
    unmapped_df.to_csv(UNMAPPED_FILE, index=False)
    print("❗ Unmapped codes found → saved at:", UNMAPPED_FILE)
else:
    pd.DataFrame(columns=["College_Code", "rows_count"]).to_csv(UNMAPPED_FILE, index=False)
    print("✅ No unmapped codes. Empty file created.")

# 7) Mapping Report
report = {
    "input_csv": str(INPUT_CSV),
    "mapping_json": str(MAPPING_JSON),
    "output_csv": str(OUTPUT_CSV),
    "total_rows": total_rows,
    "mapped_rows": int(n_mapped),
    "unmapped_rows": int(n_unmapped),
    "unique_codes_in_data": int(df["College_Code"].nunique()),
    "codes_in_mapping_json": int(len(college_map)),
}
pd.DataFrame(list(report.items()), columns=["metric", "value"]).to_csv(REPORT_FILE, index=False)

print("\nReport saved →", REPORT_FILE)

# 8) Quick Summary
print("\n=== QUICK MAPPING SUMMARY ===")
print("Total rows:", total_rows)
print("Mapped rows:", n_mapped)
print("Unmapped rows:", n_unmapped)
print("Unique college codes in dataset:", df["College_Code"].nunique())
print("College codes in mapping JSON:", len(college_map))

print("\nSample mapped rows:")
display(df[df["College_Name"].notna()].head(5)[["College_Code", "College_Name"]])

if n_unmapped > 0:
    print("\n❗ Sample UNMAPPED rows:")
    display(df[df["College_Name"].isna()].head(10))
else:
    print("\nNo unmapped rows.")


Notebook Directory: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\notebooks
Base Directory: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2
Data Directory: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\processed_data

Loading CSV: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\processed_data\KCET_split_cleaned.csv
Rows, columns: (215058, 12)
Loading JSON mapping: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\processed_data\college_mapping_2024.json

Mapped CSV saved → D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\processed_data\KCET_split_mapped.csv
❗ Unmapped codes found → saved at: D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\processed_data\kcet_unmapped_college_codes.csv

Report saved → D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\processed_data\kcet_college_mapping_report.csv

=== QUICK M

,College_Code,College_Name
0,E001,University of Visvesvaraya College of Engineer...
1,E001,University of Visvesvaraya College of Engineer...
2,E001,University of Visvesvaraya College of Engineer...
3,E001,University of Visvesvaraya College of Engineer...
4,E001,University of Visvesvaraya College of Engineer...



❗ Sample UNMAPPED rows:


,College_Code,College_Name,Category,Branch,Cutoff_Rank,Year,Round,Exam_Type,Rank_Scaled,Branch_Norm,College_Clean,College_Final,College_Name_mapped,College_Name_original
1052,E010,NaN,2AG,CE Civil,130371.0,2020,1,KCET,0.8495217119324402,CE Civil,islamia institute of technology bangalore,islamia institute of technology bangalore,NaN,Islamia Institute of Technology Bangalore
1053,E010,NaN,3BG,CE Civil,114331.0,2020,1,KCET,0.7450020851795861,CE Civil,islamia institute of technology bangalore,islamia institute of technology bangalore,NaN,Islamia Institute of Technology Bangalore
1054,E010,NaN,GM,CE Civil,101234.0,2020,1,KCET,0.6596595944325705,CE Civil,islamia institute of technology bangalore,islamia institute of technology bangalore,NaN,Islamia Institute of Technology Bangalore
1055,E010,NaN,GMR,CE Civil,121921.0,2020,1,KCET,0.7944599384872022,CE Civil,islamia institute of technology bangalore,islamia institute of technology bangalore,NaN,Islamia Institute of Technology Bangalore
1056,E010,NaN,SCG,CE Civil,147777.0,2020,1,KCET,0.962942449043424,CE Civil,islamia institute of technology bangalore,islamia institute of technology bangalore,NaN,Islamia Institute of Technology Bangalore
1057,E010,NaN,STG,CE Civil,120170.0,2020,1,KCET,0.7830500964395558,CE Civil,islamia institute of technology bangalore,islamia institute of technology bangalore,NaN,Islamia Institute of Technology Bangalore
1058,E010,NaN,2AG,CS Computers,42207.0,2020,1,KCET,0.2750286712193087,CS Computers,islamia institute of technology bangalore,islamia institute of technology bangalore,NaN,Islamia Institute of Technology Bangalore
1059,E010,NaN,2BG,CS Computers,36533.0,2020,1,KCET,0.2380558306834176,CS Computers,islamia institute of technology bangalore,islamia institute of technology bangalore,NaN,Islamia Institute of Technology Bangalore
1060,E010,NaN,3AG,CS Computers,31948.0,2020,1,KCET,0.2081791169264452,CS Computers,islamia institute of technology bangalore,islamia institute of technology bangalore,NaN,Islamia Institute of Technology Bangalore
1061,E010,NaN,3BG,CS Computers,29623.0,2020,1,KCET,0.1930289839962466,CS Computers,islamia institute of technology bangalore,islamia institute of technology bangalore,NaN,Islamia Institute of Technology Bangalore


In [1]:
# Remove discontinued / unmapped college codes from KCET_split_cleaned.csv

import pandas as pd
from pathlib import Path
import json

# Directories (relative to Stage1EDA.ipynb)
BASE = Path(".").resolve().parent   # Version2/
DATA_DIR = BASE / "data" / "processed_data"

INPUT_CSV = DATA_DIR / "KCET_split_cleaned.csv"
MAPPING_JSON = DATA_DIR / "college_mapping_2024.json"
OUTPUT_CSV = DATA_DIR / "KCET_split_cleaned_filtered.csv"

# Load data
df = pd.read_csv(INPUT_CSV, dtype=str)

# Load mapping to know valid codes
with open(MAPPING_JSON, "r", encoding="utf-8") as f:
    mapping = json.load(f)
valid_codes = set(mapping.keys())

print("Total rows before filtering:", len(df))

# Filter: keep only rows where College_Code appears in mapping
df_filtered = df[df["College_Code"].isin(valid_codes)].copy()

print("Total rows after filtering:", len(df_filtered))
print("Rows removed:", len(df) - len(df_filtered))

# Save output
df_filtered.to_csv(OUTPUT_CSV, index=False)
print("\nFiltered dataset saved →", OUTPUT_CSV)

# Show removed codes for confirmation
removed_codes = sorted(set(df["College_Code"]) - valid_codes)
print("\nRemoved discontinued codes:", removed_codes)


Total rows before filtering: 215058
Total rows after filtering: 213183
Rows removed: 1875

Filtered dataset saved → D:\Courses\Global Academy of Technology\kcet-college-pred\Version2\data\processed_data\KCET_split_cleaned_filtered.csv

Removed discontinued codes: ['E010', 'E030', 'E053', 'E089', 'E117', 'E183', 'E195', 'E208', 'E217', 'E219', 'E233']
